# Packages

<div style="border: 1px solid #e0e0e0; border-radius: 2px; padding: 5px; font-family: sans-serif; background-color: #fafafa;">
  <div style="background-color: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 4px; padding: 12px; font-family: monospace; position: relative; line-height: 1.6;">
    <code style="color: #212121;">pip install ipywidgets matplotlib numpy ipympl ipykernel</code><br>
  </div>
</div>

In [1]:
import io
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# 1. Wave Equation - Fixed ends

Initial Boundary Value Problem:

$$\frac{\partial^2 u}{\partial t^2} = V_s^2 \frac{\partial^2 u}{\partial z^2}\quad, \quad t >0 \:, \: 0 < z < H$$

* ICs: 
    * $u(0,t) = 0$, $\quad t>0$

    * $u(H,t) = 0$, $\quad t>0$
* BCs: 
    * $u(z,0) = h(z) = e^{-150(z-z_0)^2}$, $\quad 0<z<H$  (inital displacement)

    * $\frac{\partial u}{\partial t} (z,0) = 0$, $\quad 0<z<H$ (initial velocity)


In [2]:
# Prevent duplicate static figure display

# Domain & Math Functions
L = 1.0
c = 1.0
x_ext = np.linspace(-L, 2 * L, 900)
x_phys = np.linspace(0, L, 300)

def f_base(x, x0):
    return np.exp(-150 * (x - x0)**2)

def odd_periodic_extension(x, x0):
    x_mod = ((x + L) % (2 * L)) - L
    return np.sign(x_mod) * f_base(np.abs(x_mod), x0)

# Initialize Figure (scaled & spaced cleanly for VS Code)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.6, 4.4), sharex=True, dpi=120)
plt.subplots_adjust(hspace=0.32, bottom=0.12, top=0.92, left=0.10, right=0.97)

# Subplot 1: Displacement on [0, L]
title_ax1 = ax1.set_title("Displacement on $[0, L]$  |  $t = 0.00$ s", fontsize=10.5)
ax1.set_ylabel("Displacement $u(z,t)$", fontsize=9.5)
ax1.tick_params(labelsize=8.5)
ax1.set_ylim(-1.1, 1.1)
ax1.grid(True, alpha=0.3)
ax1.axvspan(-L, 0, color="gray", alpha=0.15)
ax1.axvspan(L, 2 * L, color="gray", alpha=0.15)
ax1.plot([0, L], [0, 0], "ko", markersize=6.5, zorder=5, label="Fixed Ends ($u=0$)")

line_ext_sum, = ax1.plot(x_ext, np.zeros_like(x_ext), color="purple", alpha=0.35, lw=1.5, label="Extended Sum")
line_phys_sum, = ax1.plot(x_phys, np.zeros_like(x_phys), color="purple", lw=2.3, label="$u(z,t)$")
ax1.legend(loc="upper right", fontsize=8)

# Subplot 2: Traveling Waves
title_ax2 = ax2.set_title("Traveling Waves", fontsize=10.5)
ax2.set_xlabel("$z$", fontsize=9.5)
ax2.set_ylabel("Amplitude", fontsize=9.5)
ax2.tick_params(labelsize=8.5)
ax2.set_xlim(-L, 2 * L)
ax2.set_ylim(-1.1, 1.1)
ax2.grid(True, alpha=0.3)
ax2.axvspan(-L, 0, color="gray", alpha=0.15, label="Odd Extension")
ax2.axvspan(L, 2 * L, color="gray", alpha=0.15)
ax2.axvline(0, color="k", linestyle=":", lw=1.2)
ax2.axvline(L, color="k", linestyle=":", lw=1.2)

line_right, = ax2.plot(x_ext, np.zeros_like(x_ext), "b--", lw=1.6, label=r"Upward: $\frac{1}{2}h(z - ct)$")
line_left,  = ax2.plot(x_ext, np.zeros_like(x_ext), "r--", lw=1.6, label=r"Downward: $\frac{1}{2}h(z + ct)$")
ax2.legend(loc="upper right", fontsize=8)

# Interactive image widget
img_widget = widgets.Image(format="png", layout=widgets.Layout(width="820px", height="auto"))

# Update Logic & Widgets
def update(t=0.0, x0=0.5):
    u_right = 0.5 * odd_periodic_extension(x_ext - c * t, x0)
    u_left  = 0.5 * odd_periodic_extension(x_ext + c * t, x0)
    u_ext   = u_right + u_left
    u_phys  = 0.5 * (odd_periodic_extension(x_phys - c * t, x0) + 
                     odd_periodic_extension(x_phys + c * t, x0))
    
    line_right.set_ydata(u_right)
    line_left.set_ydata(u_left)
    line_ext_sum.set_ydata(u_ext)
    line_phys_sum.set_ydata(u_phys)
    
    title_ax1.set_text(f"Displacement on $[0, L]$  |  $t = {t:.2f}$ s")
    
    # Render into memory buffer 
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    img_widget.value = buf.getvalue()

play = widgets.Play(value=0, min=0, max=200, step=1, interval=40)
slider_t = widgets.FloatSlider(value=0.0, min=0.0, max=2.0, step=0.01, description="Time t (s)", layout=widgets.Layout(width="310px"))
slider_x0 = widgets.FloatSlider(value=0.5, min=0.3, max=0.7, step=0.05, description="Pulse z0", layout=widgets.Layout(width="310px"))

def on_play_change(change):
    slider_t.value = change["new"] * 0.01

def on_param_change(change):
    update(slider_t.value, slider_x0.value)

play.observe(on_play_change, names="value")
slider_t.observe(on_param_change, names="value")
slider_x0.observe(on_param_change, names="value")

# Render initial frame and display
update(0.0, 0.5)
controls = widgets.HBox([play, slider_t, slider_x0])
display(widgets.VBox([controls, img_widget]))


# 2. Transfer function: Undamped elastic soil on rigid bedrock

Initial Boundary Value Problem:

$$\frac{\partial^2 u}{\partial t^2} = V_s^2 \frac{\partial^2 u}{\partial z^2}\quad, \quad t >0 \:, \: < z < H$$

* ICs: 
    * $u(z,0)= 0$, $\quad 0 < z < H$

    * $\frac{\partial u}{\partial t}(z,0) = 0$, $\quad 0 < z < H$
* BCs: 
    * $\frac{\partial u}{\partial z}(0,t) = 0$, $\quad t>0$

    * $u(H,t) = U_0 \cos(\omega t)$, $\quad t>0$

Transfer function:

$$|F_1(\omega)| = \frac{1}{\left|\cos\left(\frac{\omega H}{V_s}\right)\right|} = \frac{1}{\left|\cos\left(\frac{2 \pi f H}{V_s}\right)\right|}$$

In [3]:
# Prevent duplicate static figure display 
plt.ioff()

# Frequency & Depth Grids
f_max = 15.0  # Hz
f = np.linspace(0, f_max, 2000)
z_norm = np.linspace(0, 1, 300)  # Normalized depth z/H (0 = surface, 1 = bedrock)

# Initialize Figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.6, 3.2), gridspec_kw={"width_ratios": [2.2, 1]}, dpi=120)
plt.subplots_adjust(wspace=0.38, bottom=0.19, top=0.83, left=0.09, right=0.97)

# Subplot 1: Amplification Function |F(f)|
title_ax1 = ax1.set_title(r"$|F_1(f)|$", fontsize=10.5)
ax1.set_xlabel(r"$f (Hz)$", fontsize=9.5)
ax1.set_ylabel(r"$|F_1(f)|$", fontsize=9.5)
ax1.tick_params(labelsize=8.5)
ax1.set_xlim(0, f_max)
ax1.set_ylim(0, 15)
ax1.grid(True, alpha=0.3)

(line_amp,) = ax1.plot(f, np.ones_like(f), color="navy", lw=1.8,)
line_f_sel = ax1.axvline(1.0, color="crimson", linestyle="--", lw=1.4, label=r"Selected $f$")
(pt_f_sel,) = ax1.plot([1.0], [1.0], "ro", markersize=5.5, zorder=5)
asymptote_lines = []
ax1.legend(loc="upper right", fontsize=8)

# Subplot 2: Relative Displacement Profile (u(z) - u_bedrock) / u_bedrock
title_ax2 = ax2.set_title(r"Depth Profile ($f = 1.0$ Hz)", fontsize=10.5)
ax2.set_xlabel(r"$(u(z) - u_{bedrock})/u_{bedrock}$", fontsize=9.5)
ax2.set_ylabel(r"Depth $z$ (m)", fontsize=9.5)
ax2.tick_params(labelsize=8.5)
ax2.set_xlim(-6, 6)
ax2.axvline(0, color="k", lw=0.9, label="Bedrock ($u_{rel}=0$)")

(line_profile,) = ax2.plot(np.zeros_like(z_norm), z_norm, color="teal", lw=2.2)
(pt_surface,) = ax2.plot( [0.0], [0.0], "ro", markersize=5.5, label="Surface ($z=0$)")
ax2.legend(loc="lower right", fontsize=8)

# Larger CSS width (820px) for comfortable viewing in VS Code
img_widget = widgets.Image(format="png", layout=widgets.Layout(width="820px", height="auto"))

# Interactive Update Logic
def update(H=30.0, Vs=200.0, f_sel=1.5):
  global asymptote_lines

  # Calculate wave number k = omega / Vs
  omega = 2 * np.pi * f
  kH = omega * H / Vs

  # Avoid division by zero at exact resonance asymptotes
  cos_val = np.cos(kH)
  amp = np.where(np.abs(cos_val) < 1e-3, np.nan, 1.0 / np.abs(cos_val))
  line_amp.set_ydata(amp)

  # Update natural frequencies (asymptotes): f_n = (2n - 1) * Vs / (4H)
  for line in asymptote_lines:
    line.remove()
  asymptote_lines.clear()

  n = 1
  while True:
    fn = (2 * n - 1) * Vs / (4 * H)
    if fn > f_max:
      break
    lbl = r"$f_n$" if n == 1 else None
    l = ax1.axvline(fn, color="orange", linestyle=":", lw=1.2, alpha=0.85, label=lbl)
    asymptote_lines.append(l)
    n += 1
  ax1.legend(loc="upper right", fontsize=8)

  # Update selected frequency marker & relative depth profile (starts at 0 at z = H)
  k_sel = (2 * np.pi * f_sel) / Vs
  z_phys = z_norm * H
  cos_kH_sel = np.cos(k_sel * H)
  if np.abs(cos_kH_sel) < 1e-3:
    cos_kH_sel = np.sign(cos_kH_sel) * 1e-3 if cos_kH_sel != 0 else 1e-3

  u_total = np.cos(k_sel * z_phys) / cos_kH_sel
  u_rel = u_total - 1.0  # Starts at 0 at the bedrock (z = H)
  amp_sel = np.abs(u_total[0])

  line_f_sel.set_xdata([f_sel, f_sel])
  pt_f_sel.set_data([f_sel], [min(amp_sel, 15.0)])

  # Keep x-axis symmetric and centered at 0
  x_bound = min(max(6.0, np.ceil(np.max(np.abs(u_rel)) * 1.15)), 16.0)
  ax2.set_xlim(-x_bound, x_bound)
  ax2.set_xticks([-x_bound, -x_bound / 2, 0, x_bound / 2, x_bound])

  ax2.set_ylim(H, 0)  # Invert y-axis so 0 is at the top (ground surface)
  line_profile.set_data(u_rel, z_phys)
  pt_surface.set_data([u_rel[0]], [0.0])
  title_ax2.set_text(
      rf"Relative Profile ($f = {f_sel:.2f}$ Hz)" + "\n" + rf"$|F_1(f)| = {amp_sel:.2f}$"
  )

  # Render to memory buffer
  buf = io.BytesIO()
  fig.savefig(buf, format="png")
  img_widget.value = buf.getvalue()


# Matched Sliders
slider_H = widgets.FloatSlider(value=30.0, min=5.0, max=100.0, step=1.0, description="H (m)", layout=widgets.Layout(width="270px"),)
slider_Vs = widgets.FloatSlider(value=200.0, min=80.0, max=600.0, step=10.0, description="Vs (m/s)", layout=widgets.Layout(width="270px"),)
slider_f = widgets.FloatSlider(value=1.5, min=0.0, max=f_max, step=0.05, description="f (Hz)", layout=widgets.Layout(width="270px"),)

def on_change(change):
  update(slider_H.value, slider_Vs.value, slider_f.value)


slider_H.observe(on_change, names="value")
slider_Vs.observe(on_change, names="value")
slider_f.observe(on_change, names="value")

# Initial render & display
update(30.0, 200.0, 1.5)
controls = widgets.HBox([slider_H, slider_Vs, slider_f])
display(widgets.VBox([controls, img_widget]))

# 2. Transfer function: Damped elastic soil on rigid bedrock

Initial Boundary Value Problem:

$$G\frac{\partial^2 u}{\partial t^2} + \eta \frac{\partial^3 u}{\partial t^2 \partial t}= \rho \frac{\partial^2 u}{\partial t^2}\quad, \quad t >0 \:, \: < z < H$$

$$\eta = \frac{2G}{\omega}\xi$$

* ICs: 
    * $u(z,0) = 0$, $\quad 0 < z < H$

    * $\frac{\partial u}{\partial t}(z,0) = 0$, $\quad 0 < z < H$
* BCs: 
    * $G \frac{\partial u}{\partial z}(0,t) + \eta \frac{\partial^2 u}{\partial z \partial t} (0,t)= 0$, $\quad t>0$

    * $u(H,t) = U_0 e^{i\omega t}$, $\quad t>0$

Transfer function:

$$|F_2(\omega)| = \frac{1}{\sqrt{\cos^2\left(\frac{\omega H}{V_s}\right)+\left(\frac{\xi \omega H}{V_s}\right)^2}}$$

In [4]:
# Prevent duplicate static figure display 
plt.ioff()

# Frequency & Depth Grids
f_max = 15.0  # Hz
f = np.linspace(0, f_max, 2000)
z_norm = np.linspace(0, 1, 300)  # Normalized depth z/H (0 = surface, 1 = bedrock)

# Initialize Figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.6, 3.2), gridspec_kw={"width_ratios": [2.2, 1]}, dpi=120,)
plt.subplots_adjust(wspace=0.38, bottom=0.19, top=0.82, left=0.09, right=0.97)

# Subplot 1: Damped Amplification Function |F2(f)|
title_ax1 = ax1.set_title(r"Damped Transfer Function $|F_2(f)|$", fontsize=10.5)
ax1.set_xlabel(r"$f$ (Hz)", fontsize=9.5)
ax1.set_ylabel(r"$|F_2(f)|$", fontsize=9.5)
ax1.tick_params(labelsize=8.5)
ax1.set_xlim(0, f_max)
ax1.set_ylim(0, 15)
ax1.grid(True, alpha=0.3)

(line_amp,) = ax1.plot(f, np.ones_like(f), color="navy", lw=1.8,)
line_f_sel = ax1.axvline(1.5, color="crimson", linestyle="--", lw=1.4, label=r"Selected $f$")
(pt_f_sel,) = ax1.plot([1.5], [1.0], "ro", markersize=5.5, zorder=5)
natural_freq_lines = []
ax1.legend(loc="upper right", fontsize=8)

# Subplot 2: Relative Displacement Profile (u(z) - u_rock) / u_rock (Centered at 0)
title_ax2 = ax2.set_title(r"Relative Profile ($f = 1.50$ Hz)", fontsize=10.5)
ax2.set_xlabel(r"$\mathrm{Re}[(u(z)-u_{\mathrm{rock}})/u_{\mathrm{rock}}]$", fontsize=9.0)
ax2.set_ylabel(r"Depth $z$ (m)", fontsize=9.5)
ax2.tick_params(labelsize=8.5)
ax2.set_xlim(-6, 6)
ax2.axvline(0, color="k", lw=0.9, label="Bedrock ($u_{rel}=0$)")

(line_profile,) = ax2.plot(np.zeros_like(z_norm), z_norm, color="teal", lw=2.2)
(line_env_pos,) = ax2.plot(np.zeros_like(z_norm), z_norm, color="teal", linestyle="--", lw=1.0, alpha=0.6, label="Amplitude $|u_{rel}|$")
(line_env_neg,) = ax2.plot(np.zeros_like(z_norm), z_norm, color="teal", linestyle="--", lw=1.0, alpha=0.6)
(pt_surface,) = ax2.plot([0.0], [0.0], "ro", markersize=5.5, label="Surface ($z=0$)")
ax2.legend(loc="lower right", fontsize=7.5)

# Larger CSS width (820px) for comfortable viewing in VS Code
img_widget = widgets.Image(format="png", layout=widgets.Layout(width="820px", height="auto"))

# Interactive Update Logic
def update(H=30.0, Vs=200.0, xi_pct=5.0, f_sel=1.5):
  global natural_freq_lines

  xi = xi_pct / 100.0  # Convert damping percentage to decimal
  Vs_star = Vs * (1.0 + 1j * xi)  # Complex shear wave velocity

  # Complex wave number k* = omega / Vs*
  omega = 2 * np.pi * f
  kH_star = omega * H / Vs_star

  # Damped amplification factor |F2(f)| = 1 / |cos(k* H)|
  cos_val = np.cos(kH_star)
  amp = np.where(np.abs(cos_val) < 1e-4, np.nan, 1.0 / np.abs(cos_val))
  line_amp.set_ydata(amp)

  # Update undamped natural frequency markers: f_n = (2n - 1) * Vs / (4H)
  for line in natural_freq_lines:
    line.remove()
  natural_freq_lines.clear()

  n = 1
  while True:
    fn = (2 * n - 1) * Vs / (4 * H)
    if fn > f_max:
      break
    lbl = r"$f_n$" if n == 1 else None
    l = ax1.axvline(
        fn, color="orange", linestyle=":", lw=1.2, alpha=0.85, label=lbl
    )
    natural_freq_lines.append(l)
    n += 1
  ax1.legend(loc="upper right", fontsize=8)

  # Update selected frequency marker & relative complex depth profile (anchored at 0 at z = H)
  k_sel_star = (2 * np.pi * f_sel) / Vs_star
  z_phys = z_norm * H
  cos_kH_sel = np.cos(k_sel_star * H)
  if np.abs(cos_kH_sel) < 1e-4:
    cos_kH_sel = 1e-4

  u_complex = np.cos(k_sel_star * z_phys) / cos_kH_sel
  amp_sel = np.abs(u_complex[0])  # Surface amplification |F2(f)| at z = 0

  # Subtract bedrock motion (1.0) so curve and envelope start at 0 at z = H
  u_rel_complex = u_complex - 1.0
  u_rel_real = np.real(u_rel_complex)
  u_rel_env = np.abs(u_rel_complex)

  line_f_sel.set_xdata([f_sel, f_sel])
  pt_f_sel.set_data([f_sel], [min(amp_sel, 15.0)])

  # Keep x-axis strictly symmetric and centered at 0
  x_bound = min(max(6.0, np.ceil(np.max(u_rel_env) * 1.15)), 16.0)
  ax2.set_xlim(-x_bound, x_bound)
  ax2.set_xticks([-x_bound, -x_bound / 2, 0, x_bound / 2, x_bound])

  ax2.set_ylim(H, 0)  # Invert y-axis so 0 is at the top (ground surface)
  line_profile.set_data(u_rel_real, z_phys)
  line_env_pos.set_data(u_rel_env, z_phys)
  line_env_neg.set_data(-u_rel_env, z_phys)
  pt_surface.set_data([u_rel_real[0]], [0.0])
  title_ax2.set_text(rf"Relative Profile ($f = {f_sel:.2f}$ Hz)" + "\n" + rf"$|F_2(f)| = {amp_sel:.2f}$")

  # Render to memory buffer
  buf = io.BytesIO()
  fig.savefig(buf, format="png")
  img_widget.value = buf.getvalue()

# Matched Sliders (4 controls arranged cleanly in a 2x2 grid)
slider_H = widgets.FloatSlider(value=30.0, min=5.0, max=100.0, step=1.0, description="H (m)", layout=widgets.Layout(width="360px"),)
slider_Vs = widgets.FloatSlider(value=200.0, min=80.0, max=600.0, step=10.0, description="Vs (m/s)", layout=widgets.Layout(width="360px"),)
slider_xi = widgets.FloatSlider(value=5.0, min=0.0, max=20.0, step=0.5, description="ξ (%)", layout=widgets.Layout(width="360px"),)
slider_f = widgets.FloatSlider(value=1.5, min=0.0, max=f_max, step=0.05, description="f (Hz)", layout=widgets.Layout(width="360px"),)


def on_change(change):
  update(slider_H.value, slider_Vs.value, slider_xi.value, slider_f.value)

slider_H.observe(on_change, names="value")
slider_Vs.observe(on_change, names="value")
slider_xi.observe(on_change, names="value")
slider_f.observe(on_change, names="value")

# Initial render & display
update(30.0, 200.0, 5.0, 1.5)
row1 = widgets.HBox([slider_H, slider_Vs])
row2 = widgets.HBox([slider_xi, slider_f])
display(widgets.VBox([row1, row2, img_widget]))

# 3. Transfer function: Undamped elastic soil on elastic bedrock

Transfer function:

$$|F_3(\omega)|_{\xi = 0} = \frac{1}{\sqrt{\cos^2\left(\frac{\omega H}{V_s}\right)+ \alpha^2 \sin^2\left(\frac{\omega H}{V_s}\right)}}$$

In [7]:
# Prevent duplicate static figure display 
plt.ioff()

# Frequency & Depth Grids
f_max = 15.0  # Hz
f = np.linspace(0, f_max, 2000)
z_norm = np.linspace(0, 1, 300)  # Normalized depth z/H (0 = surface, 1 = bedrock)

# Initialize Figure
fig, ax = plt.subplots(1, 1, figsize=(7.6, 3.2), dpi=120,)
plt.subplots_adjust(wspace=0.38, bottom=0.19, top=0.82, left=0.09, right=0.97)

# Subplot 1: Damped Amplification Function |F3(f)|
title_ax = ax.set_title(r"Undamped Transfer Function with elastic bedrock $|F_3(f)|$", fontsize=10.5)
ax.set_xlabel(r"$f$ (Hz)", fontsize=9.5)
ax.set_ylabel(r"$|F_3(f)|$", fontsize=9.5)
ax.tick_params(labelsize=8.5)
ax.set_xlim(0, f_max)
ax.set_ylim(0, 15)
ax.grid(True, alpha=0.3)

(line_amp,) = ax.plot(f, np.ones_like(f), color="navy", lw=1.8,)
line_f_sel = ax.axvline(1.5, color="crimson", linestyle="--", lw=1.4, label=r"Selected $f$")
(pt_f_sel,) = ax.plot([1.5], [1.0], "ro", markersize=5.5, zorder=5)
natural_freq_lines = []
ax.legend(loc="upper right", fontsize=8)

# Larger CSS width (820px) for comfortable viewing in VS Code
img_widget = widgets.Image(format="png", layout=widgets.Layout(width="820px", height="auto"))

# Interactive Update Logic
def update(H=30.0, Vs=200.0, alpha=0.1, f_sel=1.5):
  global natural_freq_lines

  # Complex wave number k* = omega / Vs*
  omega = 2 * np.pi * f
  kH = omega * H / Vs

  # Damped amplification factor |F3(f)| = 1 / |cos**2(k* H) + alpha**2 * sin**2(k* H)|
  cos_val = np.cos(kH)
  sin_val = np.sin(kH)
  amp = np.where(np.abs(cos_val**2 + alpha**2 * sin_val**2) < 1e-4, np.nan, 1.0 / np.abs(cos_val**2 + alpha**2 * sin_val**2))
  line_amp.set_ydata(amp)

  # Update undamped natural frequency markers: f_n = (2n - 1) * Vs / (4H)
  for line in natural_freq_lines:
    line.remove()
  natural_freq_lines.clear()

  n = 1
  while True:
    fn = (2 * n - 1) * Vs / (4 * H)
    if fn > f_max:
      break
    lbl = r"$f_n$" if n == 1 else None
    l = ax.axvline(
        fn, color="orange", linestyle=":", lw=1.2, alpha=0.85, label=lbl
    )
    natural_freq_lines.append(l)
    n += 1
  ax.legend(loc="upper right", fontsize=8)
  # Update selected frequency marker & relative complex depth profile (anchored at 0 at z = H)
  k_sel = (2 * np.pi * f_sel) / Vs
  cos_sel = np.cos(k_sel * H)
  sin_sel = np.sin(k_sel * H)
  amp_sel = np.where(np.abs(cos_sel**2 + alpha**2 * sin_sel**2) < 1e-4, np.nan, 1.0 / np.abs(cos_sel**2 + alpha**2 * sin_sel**2))

  line_f_sel.set_xdata([f_sel, f_sel])
  pt_f_sel.set_data([f_sel], [min(amp_sel, 15.0)])
  title_ax.set_text(rf"Undamped Transfer Function with elastic bedrock $|F_3(f)|$" + "\n" + rf"$|F_3({f_sel:.2f})| = {amp_sel:.2f}$")

  # Render to memory buffer
  buf = io.BytesIO()
  fig.savefig(buf, format="png")
  img_widget.value = buf.getvalue()

# Matched Sliders (4 controls arranged cleanly in a 2x2 grid)
slider_H = widgets.FloatSlider(value=30.0, min=5.0, max=100.0, step=1.0, description="H (m)", layout=widgets.Layout(width="360px"),)
slider_Vs = widgets.FloatSlider(value=200.0, min=80.0, max=600.0, step=10.0, description="Vs (m/s)", layout=widgets.Layout(width="360px"),)
slider_alpha = widgets.FloatSlider(value=0.1, min=0.1, max=1.0, step=0.01, description="α", layout=widgets.Layout(width="360px"),)
slider_f = widgets.FloatSlider(value=1.5, min=0.0, max=f_max, step=0.05, description="f (Hz)", layout=widgets.Layout(width="360px"),)


def on_change(change):
  update(slider_H.value, slider_Vs.value, slider_alpha.value, slider_f.value)

slider_H.observe(on_change, names="value")
slider_Vs.observe(on_change, names="value")
slider_alpha.observe(on_change, names="value")
slider_f.observe(on_change, names="value")

# Initial render & display
update(30.0, 200.0, 0.1, 1.5)
row1 = widgets.HBox([slider_H, slider_Vs])
row2 = widgets.HBox([slider_alpha, slider_f])
display(widgets.VBox([row1, row2, img_widget]))